# Imports

In [151]:
import json

import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error

# Fixed Variables

In [152]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

INPUT_DATA_DIR = DATA_DIR / "input"
OUTPUT_DATA_DIR = DATA_DIR / "output"

input_path = INPUT_DATA_DIR / "04_input.csv"

OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

output_json_metrics = OUTPUT_DATA_DIR / "metrics.json"
output_plot = OUTPUT_DATA_DIR / "forecast_plot.png"
output_feature_importances = OUTPUT_DATA_DIR / "feature_importance.csv"

# Read Data

In [153]:
df = pd.read_csv(input_path)

In [154]:
df["fecha"] = pd.to_datetime(df["fecha"])
df = df.sort_values("fecha").reset_index(drop=True)

# Feature Engineering

In [5]:
df["day_of_week"] = df["fecha"].dt.dayofweek
df["month"] = df["fecha"].dt.month
df["day_of_year"] = df["fecha"].dt.dayofyear

In [6]:
df["lag_1"] = df["demand"].shift(1)
df["lag_2"] = df["demand"].shift(2)
df["lag_3"] = df["demand"].shift(3)
df["lag_7"] = df["demand"].shift(7)
df["lag_14"] = df["demand"].shift(14)
df["lag_21"] = df["demand"].shift(21)
df["lag_30"] = df["demand"].shift(30)

In [7]:
# Rolling means
df["rolling_mean_7"] = df["demand"].shift(1).rolling(7).mean()

In [8]:
df = df.dropna().reset_index(drop=True)

In [9]:
train = df.iloc[:-30].copy()
test = df.iloc[-30:].copy()

In [10]:
features = [
    "day_of_week",
    "month",
    "day_of_year",
    # "is_weekend",
    "is_holiday",
    "tmed",
    "tmin",
    "tmax",
    "prec",
    "velmedia",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_7",
    "lag_14",
    "lag_21",
    "lag_30",
    "rolling_mean_7",
]

# Model Training

In [ ]:
model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=20,
    max_depth=-1,
    random_state=42,
    verbosity=-1,
)

model.fit(
    train[features],
    train["demand"],
)

,num_leaves,20
,learning_rate,0.05
,n_estimators,300
,random_state,42
,verbosity,-1
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [12]:
test["prediction"] = df["demand"].shift(7).iloc[-30:].values

test["prediction_ml"] = model.predict(
    test[features]
)

# Outputs

In [13]:
mae_baseline = mean_absolute_error(
    test["demand"],
    test["prediction"],
)

mape_baseline = mean_absolute_percentage_error(
    test["demand"],
    test["prediction"],
)

mae = mean_absolute_error(
    test["demand"],
    test["prediction_ml"],
)

mape = mean_absolute_percentage_error(
    test["demand"],
    test["prediction_ml"],
)

print(f"MAE baseline : {mae_baseline:.2f}")
print(f"MAPE baseline: {mape_baseline:.2%}")
print(f"MAE modelo : {mae:.2f}")
print(f"MAPE modelo: {mape:.2%}")

MAE baseline : 55977.28
MAPE baseline: 86.79%
MAE modelo : 48574.45
MAPE modelo: 82.85%


In [ ]:
plt.figure(figsize=(15,5))

plt.plot(test["fecha"], test["demand"], label="Real")
plt.plot(test["fecha"], test["prediction"], label="Weekly Baseline")
plt.plot(test["fecha"], test["prediction_ml"], label="Random Forest")

plt.title("Electricity demand forecast")
plt.xlabel("Date")
plt.ylabel("Demand")
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(
    output_plot,
    dpi=150,
    bbox_inches="tight",
)

C:\Users\jortialo\AppData\Local\Temp\ipykernel_2568\1232178035.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
feature_importance = (
    pd.DataFrame({
        "feature": features,
        "importance": model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance.to_csv(
    output_feature_importances,
    index=False,
)

,feature,importance
0,lag_1,622
1,lag_3,546
2,tmax,439
3,lag_21,392
4,lag_30,380
5,rolling_mean_7,363
6,day_of_year,363
7,lag_2,351
8,lag_14,346
9,tmin,343


In [16]:
metrics = {
    "baseline_mae": float(mae_baseline),
    "baseline_mape": float(mape_baseline),
    "model_mae": float(mae),
    "model_mape": float(mape),
}
with open(output_json_metrics, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=4)